# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1EyWl33ZyAWuyMKXz_mr9aaNu3JF8e8vS?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab, instalar **Ollama** (el programa que descarga y ejecuta los modelos) y descargar el modelo que vamos a usar. La librería `ollama` es el cliente oficial en Python para hablar con ese servidor.

### **ACTIVAR LA GPU**

En el menú de Colab ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución** y selecciona **T4 GPU**. Luego ejecuta la celda para confirmar que Colab te asignó la GPU.

In [1]:
# Verificar que Colab asignó una GPU

!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


### **INSTALAR OLLAMA**

A diferencia de una API en la nube, aquí **el modelo corre dentro de tu propia sesión de Colab**: no hay *API key* ni *Secrets*. Ollama se instala en la máquina virtual, se arranca como un servidor local (`localhost:11434`) y desde Python le enviamos los prompts.

**IMPORTANTE:** la sesión de Colab es temporal. Si se reinicia o se desconecta, hay que volver a ejecutar las celdas de instalación, servidor y descarga del modelo.

In [2]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)

!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")

Ollama instalado.


In [3]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda

import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version

Servidor de Ollama listo.
ollama version is 0.40.11


### **DESCARGAR EL MODELO**

Vamos a trabajar con **Llama 3.1 de 8B**. Guardamos el nombre en la variable `MODELO` para usar *exactamente la misma etiqueta* al descargar y al llamar al modelo; si no coinciden carácter por carácter, Ollama devuelve error.

In [4]:
# Descargar el modelo Llama (la primera vez tarda unos minutos)

MODELO = "llama3.1:8b"

!ollama pull {MODELO}
!ollama list


NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    


### **CLIENTE DE PYTHON**

In [5]:
# Instalar el cliente de Ollama y conectarlo al servidor local

!pip install ollama -q

import ollama

client = ollama.Client(host="http://localhost:11434")
print("Cliente de Ollama inicializado correctamente.")

Cliente de Ollama inicializado correctamente.


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [23]:
# Prompt de clasificación en modo zero-shot

prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'El envío llegó tarde pero el producto es excelente.' Respuesta muy breve y corta."

response_zero = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": prompt_zero_shot}]
)

print("Zero-shot:", response_zero.message.content)

Zero-shot: Negativo


In [19]:
# Prompt de clasificación en modo few-shot

prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.

Reseña: "Me encantó, llegó rápido y en perfecto estado."
Sentimiento: Positivo

Reseña: "Nunca llegó mi pedido, pésimo servicio."
Sentimiento: Negativo

Reseña: "El envío llegó tarde pero el producto es excelente."
Sentimiento:"""

response_few = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": prompt_few_shot}]
)

print("Few-shot:", response_few.message.content)
# El formato few-shot suele acotar mejor la salida a una sola palabra de la categoría esperada

Few-shot: El sentimiento de esta reseña es Mixto.


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [24]:
# Razonamiento paso a paso (chain-of-thought)

problema = (
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": problema}]
)

print(response_cot.message.content)
# problema_directo = problema.split(".")[-2]  # variante: pedir solo el número, sin razonamiento, y comparar qué tan seguido se equivoca

Para resolver este problema, necesitamos encontrar la distancia recorrida por el primer tren en 2 horas y luego determinar cuánto tiempo tarda el segundo tren en cubrir esa distancia.

**Paso 1: Calcule la distancia recorrida por el primer tren en 2 horas.**

La velocidad del primer tren es 80 km/h. En 2 horas recorrerá:

80 km/h x 2 horas = 160 km

**Paso 2: Calcule cuánto tarda el segundo tren en cubrir la distancia recorrida por el primer tren.**

La velocidad del segundo tren es 120 km/h. Para encontrar cuánto tarda en cubrir la distancia de 160 km, usamos la fórmula:

Tiempo = Distancia / Velocidad

Substituyendo los valores obtenemos:

Tiempo = 160 km / 120 km/h = 1,33 horas (aproximadamente 1 hora y 20 minutos)

Por lo tanto, el segundo tren tarda aproximadamente 1 hora y 20 minutos en alcanzar al primero.

La respuesta final es 1 hora y 20 minutos.


### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [25]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina

prompt_desconocido = (
    "¿Cuál fue el resultado de la final del hackathon interno de DEV.F del 14 de agosto de "
    "2026? Respuesta muy breve y corta."
)

response_alucinacion = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": prompt_desconocido}]
)

print(response_alucinacion.message.content)
# Por más segura que suene la respuesta, el modelo no tiene forma de saber esto: es una alucinación

Lo siento, pero no tengo información sobre el resultado de una final de hackathon interno de DEV.F en agosto de 2026. Mi base de datos se limita a información hasta 2023 y no tengo acceso a futuras eventos o resultados específicos. Si tienes más detalles o necesitas información general sobre hackathones, estaré encantado de ayudarte.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [26]:
# Instalar sentence-transformers

!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

In [45]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings

modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su "
    "empaque original.",
    "Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de "
    "envío de regreso.",
    "Los productos en oferta o liquidación no son elegibles para devolución, solo para "
    "cambio de talla.", "las pelotas cuentan $100", "se abre de lunes a viernes"
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embeddings generados: (5, 384)


In [46]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante

def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "¿abren los dias martes?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)

Fragmento recuperado: se abre de lunes a viernes


In [48]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG

prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": prompt_rag}]
)

print(response_rag.message.content)
# Compara esta respuesta contra lo que el modelo diría sin el fragmento: sin RAG probablemente improvisaría una política genérica

Sí, abren los días martes.


## **CHALLENGE: ASISTENTE DE POLÍTICAS CON RAG**

Una vez visto el ***Hands-On: Prompt Engineering y Sistemas RAG***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño asistente con RAG sobre un documento propio, comparando la respuesta **con RAG** contra la respuesta **sin RAG** para la misma pregunta. En esta solución se usa como base de conocimiento el propio reglamento de evaluación del curso IA Aplicada con Llama.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key:** Prepara el entorno de configuración del entorno con **Ollama** activando la **T4 GPU** e inicializa el cliente.

In [49]:
# Verificar que Colab asignó una GPU

!nvidia-smi --query-gpu=name,memory.total --format=csv


name, memory.total [MiB]
Tesla T4, 15360 MiB


In [50]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)

!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")


Ollama instalado.


In [51]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda

import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version


Servidor de Ollama listo.
ollama version is 0.40.11


In [52]:
# Descargar el modelo Llama e inicializar el cliente

MODELO = "llama3.1:8b"

!ollama pull {MODELO}
!ollama list

!pip install ollama -q

import ollama

client = ollama.Client(host="http://localhost:11434")
print("Cliente de Ollama inicializado correctamente.")



NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    
Cliente de Ollama inicializado correctamente.


**2. Define tu base de conocimiento y genera embeddings:**

* Instala e importa la librería de  `sentence-transformers` junto con `numpy`.

* Construye una lista llamada `documentos` con 3 fragmentos de un documento real de tu propio contexto (reglamento, políticas, FAQs, etc.) y genera sus embeddings.

In [53]:
# Instalar e importar librerías

!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np


In [54]:
# Definir la lista documentos y generar sus embeddings

modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "La evaluación del curso se compone de 70% de actividades de e-learning y 30% de los hackathones.",
    "Para certificarse, el estudiante necesita 80% de asistencia síncrona recomendada, 80% de interacción con el curso y 80% del contenido e-learning completado.",
    "Para la certificación es suficiente entregar como mínimo 1 de los 2 proyectos de hackathon; no es obligatorio entregar ambos."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embeddings generados: (3, 384)


**2. Recupera el fragmento relevante:** Define una función `buscar_fragmento(pregunta)` que calcule la similitud coseno y regrese el fragmento más relevante para una pregunta dada.

In [55]:
# Definir la función buscar_fragmento

def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]


**3. Genera la respuesta sin RAG:** Envía una pregunta real sobre tu documento directamente al modelo (sin ningún fragmento de contexto) y guarda la respuesta en `respuesta_sin_rag`.

In [56]:
# Consultar la pregunta sin RAG y guardar el resultado en respuesta_sin_rag

pregunta = "¿Necesito entregar los dos hackathones para poder certificarme en el curso?"

response_sin_rag = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": pregunta}]
)

respuesta_sin_rag = response_sin_rag.message.content

print("Respuesta sin RAG:")
print(respuesta_sin_rag)


Respuesta sin RAG:
Para certificarte en un curso de hackathons, el número de hackathones que debes entregar puede variar dependiendo del programa o plataforma que estés utilizando. Sin embargo, en general, se espera que los participantes de hackathons completen y entreguen al menos un proyecto.

Algunos programas pueden tener requisitos específicos para la entrega de proyectos, como:

1.  **Compleción de un solo hackathón**: Algunos cursos pueden exigir que completes solo un hackathón para certificarte. En este caso, el contenido y la compleción del hackathón serán los que decidan si cumpliste con los requisitos para la certificación.

2.  **Compleción de múltiples hackathones**: Otros programas pueden requerir que completes un cierto número de hackathones para obtener la certificación. Por ejemplo, un curso puede pedir que completes un mínimo de 5 o 10 hackathones para ser certificado.

3.  **Nivel de compleción o desempeño**: Algunos cursos pueden evaluar tu desempeño en los hackatho

**4. Genera la respuesta con RAG:** Recupera el fragmento relevante con tu función y envía la pregunta junto con ese fragmento al modelo. Guarda la respuesta en `respuesta_con_rag`.

In [ ]:
# Consultar la pregunta con RAG y guardar el resultado en respuesta_con_rag




Fragmento recuperado:
Para la certificación es suficiente entregar como mínimo 1 de los 2 proyectos de hackathon; no es obligatorio entregar ambos.

Respuesta con RAG:
No, solo debes entregar 1 de los 2 hackathones.


**5. Compara y concluye:** Imprime ambas respuestas y concluye cuál de las dos evitó mejor una alucinación o dio una respuesta más precisa.

In [ ]:
# Mostrar ambas respuestas para comparar




PREGUNTA
¿Necesito entregar los dos hackathones para poder certificarme en el curso?

RESPUESTA SIN RAG
No tengo información sobre los requisitos específicos del curso en el que estás inscrito. Si necesitas ayuda con un hackathon o con el curso en general, te recomendaría que consultes la documentación del curso o comuniques con el instructor o administrador del curso para obtener la información más actualizada sobre los requisitos de certificación.

RESPUESTA CON RAG
No, solo debes entregar 1 de los 2 hackathones.

CONCLUSIÓN
La respuesta con RAG es más confiable para esta pregunta porque el modelo recibe un fragmento específico del reglamento del curso. El fragmento indica que basta con entregar como mínimo 1 de los 2 hackathones para certificarse.
